# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/llama-nemotron-embed-vl-1b-v2`
- Generación: `nvidia/nemotron-3.5-lightning-30b-a3b`

---
**Versión modificada para la entrega del workshop.** Después del Bonus se agregan las secciones de las tareas **B2–B5** y de los retos **C3, C4, C5 y C6**. Cada sección imprime tablas en Markdown listas para pegar en `RESPUESTAS.md`.

> Nota: la extracción de MarkItDown pega muchas palabras (por ejemplo `WeusedtheAdamoptimizer`). Lo tengo en cuenta en las comprobaciones automáticas, que ignoran los espacios.

In [1]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests


In [2]:
import os
import getpass
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-lightning-30b-a3b"
EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2"


from langchain_core.embeddings import Embeddings


class RetryEmbeddings(Embeddings):
    """Reintenta las llamadas de embeddings: el endpoint de NVIDIA a veces responde 502 tras 30 s."""

    def __init__(self, inner, attempts: int = 12, wait: float = 1.0, timeout: float = 12.0):
        self.inner, self.attempts, self.wait = inner, attempts, wait
        # Las respuestas buenas llegan en < 1 s; las que fallan se cuelgan 30 s.
        # Un timeout corto corta antes las colgadas y reintenta.
        if hasattr(inner, "_client"):
            inner._client.timeout = timeout

    def _retry(self, fn, *args):
        import time
        for attempt in range(1, self.attempts + 1):
            try:
                return fn(*args)
            except Exception as error:
                if attempt == self.attempts:
                    raise
                print(f"  ↻ embeddings: {str(error).splitlines()[0][:30]} · reintento {attempt}")
                time.sleep(min(self.wait * attempt, 5))

    def embed_documents(self, texts):
        # lotes pequeños: si un lote falla solo se repite ese lote
        out = []
        for i in range(0, len(texts), 32):
            out += self._retry(self.inner.embed_documents, texts[i:i + 32])
        return out

    def embed_query(self, text):
        # caché: las mismas preguntas se repiten en varios experimentos (B2, C4, C5)
        if not hasattr(self, "_cache"):
            self._cache = {}
        if text not in self._cache:
            self._cache[text] = self._retry(self.inner.embed_query, text)
        return self._cache[text]

print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


In [3]:
from markitdown import MarkItDown

DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return

    response = requests.get(url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")


for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")


↪ Ya existe: attention_paper.pdf
↪ Ya existe: rag_paper.pdf


✅ attention_paper.pdf → 40,174 caracteres
✅ rag_paper.pdf       → 69,121 caracteres

Inicio del paper de Attention:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog...


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def pdf_to_documents(text: str, title: str, source: str,
                     chunk_size: int = 900, chunk_overlap: int = 120) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)


Attention → 55 chunks
RAG       → 95 chunks
Total     → 150 chunks

Ejemplo de metadatos: {'title': 'Attention Is All You Need', 'source': 'attention_paper.pdf', 'chunk_index': 0}


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [5]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

embeddings = RetryEmbeddings(NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    truncate="END",
))
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = vectorstore.add_documents(all_docs)

print(f"✅ {len(ids)} chunks indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


...

  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 5


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 6


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 7


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 8


✅ 150 chunks indexados en Qdrant
✅ Dimensión de embeddings: 2048


---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [6]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    # Nemotron 3.5 razona por defecto y ese razonamiento consume max_completion_tokens;
    # sin desactivarlo, muchas respuestas llegan vacías.
    model_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=800,
).with_retry(stop_after_attempt=4)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


SYSTEM_PROMPT_ORIGINAL = (
    "Answer using only the supplied academic-paper excerpts. "
    "If the excerpts do not contain enough evidence, say so explicitly. "
    "Answer in the same language as the question and be concise. "
    "Mention the paper title supporting the answer."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT_ORIGINAL),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = rag_chain.invoke(test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


Pregunta: What is the Transformer architecture?
Respuesta: The Transformer architecture is a network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. It uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. Each encoder layer has two sub-layers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network, with residual connections and layer normalization. The model achieves state-of-the-art results in machine translation tasks while being more parallelizable and requiring significantly less training time.


---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [7]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


Q: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?
A: 28.4


<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [8]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


Q: How does RAG combine parametric and non-parametric memory?
A: RAG combines parametric and non-parametric memory by using a pre-trained seq2seq model as the parametric memory and a dense vector index of Wikipedia accessed via a pre-trained neural retriever as the non-parametric memory, trained end-to-end with a probabilistic model.


<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [9]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


Q: What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?
A: The scaling factor sqrt(dk) is used to counteract the effect of large dot product magnitudes when dk is large, which can push the softmax function into regions with extremely small gradients, hindering training.


<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [10]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

Q: What are the two RAG model variants proposed in the paper and how do they differ?
A: The paper proposes two RAG model variants: RAG-Sequence and RAG-Token. RAG-Sequence uses the same retrieved document to predict every target token, marginalizing over a single latent document for the entire sequence. In contrast, RAG-Token allows drawing a different latent document for each target token, enabling the generator to choose content from several documents when producing an answer.


<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [11]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


Q: How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?
A: The base Transformer model uses 8 attention heads, with key and value dimensions of 64 per head.


<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [12]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = rag_chain.invoke(q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


Q: The RAG paper uses FAISS for fast retrieval with an HNSW approximation. The Attention paper proposes the Transformer architecture. What seq2seq model does RAG use as its generator component, and is it based on the Transformer?

A: RAG uses a seq2seq generator based on the Transformer architecture. The paper states that the generator is parametrized as p_θ(y_i | x, z, y_{1:i-1}), which generates the current token based on the context of the previous i-1 tokens, the original input x, and the retrieved passage z. This formulation is consistent with the Transformer's seq2seq design. The paper also mentions that the generator is trained end-to-end with the retriever, and the implementation is based on HuggingFace Transformers, which are Transformer-based. 

Paper: Retrieval-Augmented Generation (RAG)

Fuentes recuperadas:
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Au

<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## Utilidades de evaluación (B1–B5)

`QUIZ` reúne las preguntas Q1–Q5 con comprobaciones automáticas por palabras clave. Una respuesta cuenta como acierto automático si contiene al menos una palabra de **cada** grupo. Es una ayuda para las 9 configuraciones de B2; en B1 la clasificación sí/parcial/no la hago leyendo la respuesta frente al ground truth.

In [13]:
import re
import time

QUIZ = {
    "Q1": (q1, [[r"28\.4"]]),
    "Q2": (q2, [[r"bart", r"seq2seq", r"parametric"], [r"wikipedia", r"dense", r"non-?parametric", r"no paramétric"]]),
    "Q3": (q3, [[r"softmax"], [r"gradient", r"gradiente"]]),
    "Q4": (q4, [[r"rag-?sequence"], [r"rag-?token"]]),
    "Q5": (q5, [[r"\b8\b", r"eight", r"ocho"], [r"\b64\b"]]),
}


def auto_correct(answer: str, groups) -> bool:
    text = answer.lower()
    return all(any(re.search(p, text) for p in group) for group in groups)


def one_line(text: str, limit: int = 220) -> str:
    text = " ".join(text.split()).replace("|", "\\|")
    return text if len(text) <= limit else text[:limit] + "…"


def retrieved_summary(retr, question: str) -> str:
    docs = retr.invoke(question)
    return ", ".join(f"{d.metadata['source'].split('_')[0]}#{d.metadata['chunk_index']}" for d in docs) or "(vacío)"


def run_quiz(chain, retr, pause: float = 1.0):
    """Devuelve [(id, respuesta, acierto_auto, chunks_recuperados)]."""
    out = []
    for qid, (question, groups) in QUIZ.items():
        answer = chain.invoke(question)
        out.append((qid, answer, auto_correct(answer, groups), retrieved_summary(retr, question)))
        time.sleep(pause)
    return out

## B1 · Tabla de resultados del quiz

Imprime la respuesta de cada pregunta y los chunks recuperados (`paper#chunk_index`). En la columna «¿Correcta?» pongo sí / parcial / no después de leer el ground truth.

In [14]:
b1 = run_quiz(rag_chain, retriever)
answers_b1 = {qid: ans for qid, ans, _, _ in b1}
print("| # | Respuesta del RAG | Auto | Chunks recuperados | ¿Correcta? | ¿Paper correcto? |")
print("|---|---|:---:|---|:---:|:---:|")
for qid, ans, ok, chunks in b1:
    print(f"| {qid} | {one_line(ans)} | {'✔' if ok else '✘'} | {chunks} | | |")

print(f"| Bonus | {one_line(answer_bonus)} | — | {retrieved_summary(retriever, q_bonus)} | | |")

| # | Respuesta del RAG | Auto | Chunks recuperados | ¿Correcta? | ¿Paper correcto? |
|---|---|:---:|---|:---:|:---:|
| Q1 | 28.4 | ✔ | attention#28, attention#1, attention#26, attention#31, attention#24 | | |
| Q2 | RAG combines parametric and non-parametric memory by using a pre-trained seq2seq model as the parametric memory and a dense vector index of Wikipedia accessed via a pre-trained neural retriever as the non-parametric memo… | ✔ | rag#8, rag#1, rag#7, rag#32, rag#50 | | |
| Q3 | The scaling factor sqrt(dk) is used to counteract the effect of dot products growing large in magnitude for large values of dk, which pushes the softmax function into regions with extremely small gradients, making traini… | ✔ | attention#11, attention#12, attention#13, attention#16, attention#10 | | |
| Q4 | The paper proposes two RAG model variants: RAG-Sequence and RAG-Token. RAG-Sequence uses the same retrieved document to predict each target token, marginalizing over a single latent document for t

---
## B2 · Experimentos con `chunk_size`, `k` y `score_threshold`

`build_store` reconstruye el índice con otro chunking (overlap ≈ 13 %) y `build_chain` arma la cadena sobre cualquier store. Para 900 se reutiliza el índice de la Parte 3, así que solo hay dos construcciones nuevas (400 y 1500).

In [15]:
def build_store(chunk_size: int, chunk_overlap: int, collection: str):
    docs = (pdf_to_documents(attention_text, "Attention Is All You Need", "attention_paper.pdf", chunk_size, chunk_overlap)
            + pdf_to_documents(rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf", chunk_size, chunk_overlap))
    if not qdrant_client.collection_exists(collection):
        qdrant_client.create_collection(
            collection_name=collection,
            vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
        )
        store = QdrantVectorStore(client=qdrant_client, collection_name=collection, embedding=embeddings)
        store.add_documents(docs)
    else:
        store = QdrantVectorStore(client=qdrant_client, collection_name=collection, embedding=embeddings)
    return store, len(docs)


def build_chain(store, k: int = 5, score_threshold: float = 0.25, system_prompt: str = SYSTEM_PROMPT_ORIGINAL):
    retr = store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": k, "score_threshold": score_threshold},
    )
    chain_prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
    ])
    chain = {"context": retr | format_docs, "question": RunnablePassthrough()} | chain_prompt | llm | StrOutputParser()
    return chain, retr


stores = {900: (vectorstore, len(all_docs))}
for size in (400, 1500):
    t0 = time.time()
    stores[size] = build_store(size, round(size * 0.133), f"papers_cs{size}")
    print(f"chunk_size={size}: {stores[size][1]} chunks indexados en {time.time() - t0:.1f}s")

  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


...

  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 3


chunk_size=400: 326 chunks indexados en 201.5s


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


chunk_size=1500: 96 chunks indexados en 30.6s


In [16]:
CONFIGS = (
    [("chunk_size", cs, dict(cs=cs, k=5, thr=0.25)) for cs in (400, 900, 1500)]
    + [("k", k, dict(cs=900, k=k, thr=0.25)) for k in (3, 5, 8)]
    + [("score_threshold", thr, dict(cs=900, k=5, thr=thr)) for thr in (0.15, 0.25, 0.40)]
)

b2_rows = []
for param, value, cfg in CONFIGS:
    store, n_chunks = stores[cfg["cs"]]
    chain, retr = build_chain(store, cfg["k"], cfg["thr"])
    results = run_quiz(chain, retr)
    hits = sum(ok for _, _, ok, _ in results)
    avg_ret = sum(len(retr.invoke(QUIZ[qid][0])) for qid in QUIZ) / len(QUIZ)
    failed = [qid for qid, _, ok, _ in results if not ok]
    b2_rows.append((param, value, cfg, n_chunks, hits, avg_ret, failed))
    print(f"{param}={value}: {hits}/5  fallos={failed}")

print("\n| Experimento | Valor | chunks en índice | chunks recuperados (prom.) | Aciertos (auto) | Fallos | Observación |")
print("|---|---:|---:|---:|---:|---|---|")
for param, value, cfg, n_chunks, hits, avg_ret, failed in b2_rows:
    print(f"| {param} | {value} | {n_chunks} | {avg_ret:.1f} | {hits}/5 | {', '.join(failed) or '—'} | |")

chunk_size=400: 5/5  fallos=[]


chunk_size=900: 5/5  fallos=[]


chunk_size=1500: 5/5  fallos=[]


k=3: 5/5  fallos=[]


k=5: 5/5  fallos=[]


k=8: 5/5  fallos=[]


score_threshold=0.15: 5/5  fallos=[]


score_threshold=0.25: 5/5  fallos=[]


score_threshold=0.4: 5/5  fallos=[]

| Experimento | Valor | chunks en índice | chunks recuperados (prom.) | Aciertos (auto) | Fallos | Observación |
|---|---:|---:|---:|---:|---|---|
| chunk_size | 400 | 326 | 5.0 | 5/5 | — | |
| chunk_size | 900 | 150 | 5.0 | 5/5 | — | |
| chunk_size | 1500 | 96 | 5.0 | 5/5 | — | |
| k | 3 | 150 | 3.0 | 5/5 | — | |
| k | 5 | 150 | 5.0 | 5/5 | — | |
| k | 8 | 150 | 8.0 | 5/5 | — | |
| score_threshold | 0.15 | 150 | 5.0 | 5/5 | — | |
| score_threshold | 0.25 | 150 | 5.0 | 5/5 | — | |
| score_threshold | 0.4 | 150 | 5.0 | 5/5 | — | |


---
## B3 · Preguntas trampa

Tres preguntas plausibles cuya respuesta **no** está en los papers. Clasifico cada respuesta como: admite que no sabe / inventa / mezcla verdad con invención.

In [17]:
TRAPS = [
    "What BLEU score did GPT-4 achieve on the WMT 2014 English-to-German task?",
    "How does RAG compare against Llama 2 70B on Natural Questions?",
    "How many GPUs were used to train RAG-Token, and how many days did training take?",
]

for question in TRAPS:
    print("Q:", question)
    print("A:", rag_chain.invoke(question))
    print("Chunks:", retrieved_summary(retriever, question), "\n")
    time.sleep(1)

Q: What BLEU score did GPT-4 achieve on the WMT 2014 English-to-German task?


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


A: The provided text does not contain information about GPT-4's BLEU score on the WMT 2014 English-to-German task. The excerpts discuss the Transformer model's performance, which achieved a BLEU score of 28.4 on that task, but GPT-4 is not mentioned.
Chunks: attention#28, attention#26, attention#1, attention#24, attention#38 



Q: How does RAG compare against Llama 2 70B on Natural Questions?


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


A: The provided excerpts do not contain enough evidence to answer the question. There is no mention of Llama 2 70B in the text, and the comparison against Natural Questions is only discussed in the context of RAG's state-of-the-art performance relative to other models like DPR, T5, and BART, without referencing Llama 2.
Chunks: rag#19, rag#29, rag#41, rag#40, rag#25 



Q: How many GPUs were used to train RAG-Token, and how many days did training take?


A: The training used 8 GPUs and took 3 days.
Chunks: rag#86, rag#41, rag#85, rag#16, rag#13 



Si alguna respuesta inventa o mezcla, se prueba este prompt más estricto. Endurece tres cosas: exige que **cada** dato numérico esté literalmente en los extractos, prohíbe usar conocimiento previo del modelo y fija una frase de salida.

In [18]:
SYSTEM_PROMPT_B3 = (
    "You answer questions about two academic papers using ONLY the excerpts in CONTEXT. "
    "Never use prior knowledge, even if you are confident. "
    "Every number, model name or result you mention must appear literally in the excerpts. "
    "If the question asks about a model, dataset or result that is not mentioned in the excerpts "
    "(for example, models published after these papers), reply exactly: "
    "\"The papers do not contain this information.\" (or its translation to the question's language) "
    "and then, in one sentence, what the excerpts DO say that is closest. "
    "Answer in the same language as the question, be concise, and cite the paper title in brackets."
)

strict_chain, _ = build_chain(vectorstore, k=5, score_threshold=0.25, system_prompt=SYSTEM_PROMPT_B3)

print("| Pregunta | Antes (prompt original) | Después (prompt B3) |")
print("|---|---|---|")
for question in TRAPS:
    before = rag_chain.invoke(question); time.sleep(1)
    after = strict_chain.invoke(question); time.sleep(1)
    print(f"| {question} | {one_line(before)} | {one_line(after)} |")

# Control: el prompt estricto no debe romper una pregunta que sí tiene respuesta.
print("\nControl Q1 con prompt B3:", strict_chain.invoke(q1))

| Pregunta | Antes (prompt original) | Después (prompt B3) |
|---|---|---|


| What BLEU score did GPT-4 achieve on the WMT 2014 English-to-German task? | The provided excerpts do not contain information about GPT-4's BLEU score on the WMT 2014 English-to-German task. The documents discuss the Transformer model's performance, which achieved a BLEU score of 28.4 on this tas… | The papers do not contain this information. The excerpts state that the Transformer(big) model achieved a BLEU score of 28.4 on the WMT 2014 English-to-German task. |


| How does RAG compare against Llama 2 70B on Natural Questions? | The provided excerpts do not contain enough evidence to answer the question. There is no mention of Llama 2 70B or a direct comparison between RAG and Llama 2 70B on the Natural Questions dataset in the given text. | The papers do not contain this information. The excerpts state that RAG sets a new state-of-the-art on all four open-domain QA tasks, including Natural Questions, and combines the generation flexibility of closed-book ap… |


| How many GPUs were used to train RAG-Token, and how many days did training take? | Training RAG-Token used 8 GPUs and took 3 days. | The papers do not contain this information. The excerpts state that training was distributed across 8, 32GB NVIDIA V100 GPUs. |



Control Q1 con prompt B3: 28.4


---
## B4 · Preguntas en español sobre papers en inglés

Para Q1, Q3 y Q4 comparo los chunks recuperados en inglés y en español (top-5 con su score) y la respuesta.

In [19]:
ES = {
    "Q1": "¿Qué puntaje BLEU obtuvo el Transformer (modelo grande) en WMT 2014 inglés-alemán?",
    "Q3": "¿Cuál es el propósito del factor de escala sqrt(dk) en el Scaled Dot-Product Attention?",
    "Q4": "¿Cuáles son las dos variantes del modelo RAG propuestas en el paper y en qué se diferencian?",
}


def top_chunks(question: str, k: int = 5):
    return [(f"{d.metadata['source'].split('_')[0]}#{d.metadata['chunk_index']}", s)
            for d, s in vectorstore.similarity_search_with_score(question, k=k)]


print("| # | Chunks EN (score) | Chunks ES (score) | Coincidencia | Auto EN | Auto ES |")
print("|---|---|---|:---:|:---:|:---:|")
answers_es = {}
for qid, q_es in ES.items():
    q_en, groups = QUIZ[qid]
    en, es = top_chunks(q_en), top_chunks(q_es)
    overlap = len({c for c, _ in en} & {c for c, _ in es})
    answers_es[qid] = rag_chain.invoke(q_es); time.sleep(1)
    fmt = lambda xs: ", ".join(f"{c} ({s:.2f})" for c, s in xs)
    print(f"| {qid} | {fmt(en)} | {fmt(es)} | {overlap}/5 | "
          f"{'✔' if auto_correct(answers_b1[qid], groups) else '✘'} | {'✔' if auto_correct(answers_es[qid], groups) else '✘'} |")

for qid, ans in answers_es.items():
    print(f"\n{qid} (ES): {ans}")

| # | Chunks EN (score) | Chunks ES (score) | Coincidencia | Auto EN | Auto ES |
|---|---|---|:---:|:---:|:---:|


| Q1 | attention#28 (0.58), attention#1 (0.49), attention#26 (0.46), attention#31 (0.41), attention#24 (0.38) | attention#28 (0.53), attention#1 (0.50), attention#26 (0.46), attention#31 (0.40), attention#40 (0.37) | 4/5 | ✔ | ✔ |


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 3


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 4


| Q3 | attention#11 (0.42), attention#12 (0.41), attention#13 (0.40), attention#16 (0.26), attention#10 (0.24) | attention#12 (0.40), attention#11 (0.40), attention#13 (0.40), attention#16 (0.25), attention#10 (0.23) | 5/5 | ✔ | ✔ |


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


| Q4 | rag#10 (0.48), rag#11 (0.47), rag#24 (0.42), rag#12 (0.38), rag#91 (0.38) | rag#10 (0.43), rag#11 (0.42), rag#24 (0.39), rag#91 (0.34), rag#12 (0.34) | 5/5 | ✔ | ✔ |

Q1 (ES): 28.4 BLEU (atención: atención_paper.pdf)

Q3 (ES): El propósito del factor de escala sqrt(dk) es contrarrestar el crecimiento de la magnitud de los productos punto cuando dk es grande, lo que empuja a la función softmax a regiones con gradientes extremadamente pequeños, dificultando el aprendizaje. Al dividir por sqrt(dk), se mantiene la estabilidad del gradiente y se mejora el rendimiento con valores grandes de dk.

Q4 (ES): Las dos variantes del modelo RAG propuestas son **RAG-Sequence** y **RAG-Token**. La diferencia principal radica en cómo se utiliza el documento recuperado para generar la secuencia objetivo:

*   **RAG-Sequence**: Utiliza el mismo documento recuperado para predecir cada token de la secuencia objetivo. El documento se trata como una variable latente única que se marginaliza para obten

Prueba de la hipótesis: si el modelo de embeddings es multilingüe, una pregunta y su traducción deberían quedar muy cerca en el espacio vectorial (coseno alto), y bastante más cerca que dos preguntas distintas en el mismo idioma.

In [20]:
import numpy as np

def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

vec = {f"{qid}-{lang}": embeddings.embed_query(q) for qid in ES for lang, q in (("EN", QUIZ[qid][0]), ("ES", ES[qid]))}
for qid in ES:
    print(f"{qid}: coseno(EN, ES) = {cos(vec[qid + '-EN'], vec[qid + '-ES']):.3f}")
print(f"Control Q1-EN vs Q4-EN = {cos(vec['Q1-EN'], vec['Q4-EN']):.3f}")

Q1: coseno(EN, ES) = 0.848
Q3: coseno(EN, ES) = 0.966
Q4: coseno(EN, ES) = 0.929
Control Q1-EN vs Q4-EN = 0.101


---
## B5 · Mi propio quiz con ground truth

Tres preguntas nuevas. Verifiqué la respuesta leyendo el texto extraído de los PDFs. La tercera exige combinar los dos papers.

In [21]:
# ── Mi pregunta 1 ────────────────────────────────────────────
mq1 = "What optimizer and learning rate schedule were used to train the Transformer, and how many warmup steps?"

answer_mq1 = rag_chain.invoke(mq1)
print(f"Q: {mq1}")
print(f"A: {answer_mq1}")
print("Chunks:", retrieved_summary(retriever, mq1))

Q: What optimizer and learning rate schedule were used to train the Transformer, and how many warmup steps?
A: The Transformer used the Adam optimizer with β₁ = 0.9, β₂ = 0.98, and ε = 10⁻⁹. The learning rate was warmed up for 4000 steps and then decreased proportionally to the inverse square root of the step number.
Chunks: attention#29, attention#25, attention#31, attention#39, attention#28


<details>
<summary>✅ Ground Truth — Mi pregunta 1</summary>

El Transformer se entrenó con **Adam** (β1 = 0.9, β2 = 0.98, ε = 10⁻⁹). La tasa de aprendizaje
crece **linealmente durante los primeros `warmup_steps` = 4000 pasos** y luego decrece proporcional a la
inversa de la raíz cuadrada del número de paso: lrate = d_model^-0.5 · min(step^-0.5, step · warmup^-1.5).
*(Fuente: Section 5.3 «Optimizer» — Attention Is All You Need)*
</details>

In [22]:
# ── Mi pregunta 2 ────────────────────────────────────────────
mq2 = "What knowledge source does RAG use as its non-parametric memory, and how is it split and indexed?"

answer_mq2 = rag_chain.invoke(mq2)
print(f"Q: {mq2}")
print(f"A: {answer_mq2}")
print("Chunks:", retrieved_summary(retriever, mq2))

  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 3


Q: What knowledge source does RAG use as its non-parametric memory, and how is it split and indexed?
A: RAG uses Wikipedia as its non-parametric memory, which is indexed as a dense vector index and accessed via a pre-trained neural retriever. Different documents can be responsible for different tokens, allowing the model to retrieve multiple documents per token during generation.
Chunks: rag#8, rag#7, rag#1, rag#32, rag#42


<details>
<summary>✅ Ground Truth — Mi pregunta 2</summary>

Un dump de **Wikipedia de diciembre de 2018**, dividido en fragmentos disjuntos de **100 palabras**
(**21 millones** de documentos). Cada fragmento se codifica con el encoder de documentos de DPR (BERT-base) y se
construye un índice **MIPS con FAISS** usando la aproximación **HNSW** para recuperación rápida.
*(Fuente: Section 3 «Experiments» y Section 2.2 — RAG paper)*
</details>

In [23]:
# ── Mi pregunta 3 ────────────────────────────────────────────
mq3 = "Which decoding strategies do the Transformer paper and the RAG paper use to generate their outputs?"

answer_mq3 = rag_chain.invoke(mq3)
print(f"Q: {mq3}")
print(f"A: {answer_mq3}")
print("Chunks:", retrieved_summary(retriever, mq3))

Q: Which decoding strategies do the Transformer paper and the RAG paper use to generate their outputs?
A: The Transformer paper uses beam search for output generation. The RAG paper uses Thorough Decoding for RAG-Sequence and standard beam decoding for RAG-Token.
Chunks: rag#10, rag#11, rag#17, rag#16, rag#12


<details>
<summary>✅ Ground Truth — Mi pregunta 3</summary>

- **Transformer:** beam search con **beam size 4** y **length penalty α = 0.6** (Section 6.1).
- **RAG:** RAG-Token se decodifica con un beam search estándar porque es un generador autorregresivo normal;
  RAG-Sequence no admite un solo beam search y usa **«Thorough Decoding»** (beam search por documento y pasadas
  extra para puntuar cada hipótesis) o **«Fast Decoding»** (asume probabilidad ≈ 0 para hipótesis que no aparecieron
  en el beam de un documento) (Section 2.5).
*(Fuente: ambos papers)*
</details>

---
# Retos

## C3 · App Gradio para los papers con filtro por paper

Usa el índice de la Parte 3. El filtro se aplica **dentro de Qdrant** (filtro sobre `metadata.source`), no después de recuperar, así que siempre llegan hasta `k` chunks del paper elegido.

In [ ]:
%pip install -qU gradio

In [24]:
import gradio as gr
from qdrant_client import models

PAPER_FILTERS = {
    "Todos": None,
    "Attention Is All You Need": "attention_paper.pdf",
    "Retrieval-Augmented Generation": "rag_paper.pdf",
}


def source_filter(choice: str):
    source = PAPER_FILTERS.get(choice)
    if source is None:
        return None
    return models.Filter(must=[models.FieldCondition(key="metadata.source", match=models.MatchValue(value=source))])


def ask_papers(question: str, choice: str, k: int = 5):
    hits = vectorstore.similarity_search_with_score(question, k=k, filter=source_filter(choice))
    hits = [(d, s) for d, s in hits if s >= 0.25]
    context = format_docs([d for d, _ in hits])
    answer = (prompt | llm | StrOutputParser()).invoke({"context": context, "question": question})
    sources = "\n".join(
        f"- **{d.metadata['title']}** · `{d.metadata['source']}` · chunk {d.metadata['chunk_index']} · score {s:.3f}"
        for d, s in hits
    ) or "Sin fragmentos sobre el umbral."
    return answer, sources


with gr.Blocks(title="RAG sobre papers") as papers_demo:
    gr.Markdown("# RAG sobre papers\nAttention Is All You Need · Retrieval-Augmented Generation")
    with gr.Row():
        q_box = gr.Textbox(label="Pregunta", lines=2, scale=4)
        paper_dd = gr.Dropdown(list(PAPER_FILTERS), value="Todos", label="Paper", scale=1)
    ask_btn = gr.Button("Preguntar", variant="primary")
    answer_md = gr.Markdown(label="Respuesta")
    sources_md = gr.Markdown(label="Fuentes")
    gr.Examples([[q_bonus, "Todos"], [q_bonus, "Retrieval-Augmented Generation"], [q_bonus, "Attention Is All You Need"]],
                inputs=[q_box, paper_dd])
    ask_btn.click(ask_papers, [q_box, paper_dd], [answer_md, sources_md])
    q_box.submit(ask_papers, [q_box, paper_dd], [answer_md, sources_md])

/usr/local/lib/python3.13/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Comparación sin interfaz: la misma pregunta con los tres valores del filtro.

In [25]:
for choice in PAPER_FILTERS:
    answer, sources = ask_papers(q_bonus, choice)
    print(f"### Filtro: {choice}\n{answer}\n{sources}\n")
    time.sleep(1)

### Filtro: Todos
RAG uses a seq2seq generator based on a pre-trained encoder-decoder model (BART), which is Transformer-based. The paper initializes the generator from a BART checkpoint and trains it end-to-end with the retriever.
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 10 · score 0.493
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 12 · score 0.477
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 13 · score 0.464
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 86 · score 0.463
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 11 · score 0.448



### Filtro: Attention Is All You Need
The RAG paper uses a seq2seq model with a Transformer-based generator.
- **Attention Is All You Need** · `attention_paper.pdf` · chunk 8 · score 0.379
- **Attention Is All You Need** · `attention_paper.pdf` · chunk 0 · score 0.374
- **Attention Is All You Need** · `attention_paper.pdf` · chunk 40 · score 0.340
- **Attention Is All You Need** · `attention_paper.pdf` · chunk 39 · score 0.330
- **Attention Is All You Need** · `attention_paper.pdf` · chunk 2 · score 0.327



### Filtro: Retrieval-Augmented Generation
The RAG paper uses a seq2seq generator based on the Transformer architecture. The generator p_θ(y_i | x, z, y_{1:i-1}) is parametrized to generate a current token based on the context of the previous i-1 tokens, the original input x, and the retrieved passage z. The paper explicitly states that the generator is trained end-to-end with the retriever, and the implementation is built on HuggingFace Transformers, which are Transformer-based. 

Paper: [Retrieval-Augmented Generation | rag_paper.pdf]
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 10 · score 0.493
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 12 · score 0.477
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 13 · score 0.464
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 86 · score 0.463
- **Retrieval-Augmented Generation** · `rag_paper.pdf` · chunk 11 · score 0.448



In [ ]:
# Lanza la app. Con debug=False la celda termina y la app sigue viva mientras el runtime esté activo.
papers_demo.queue().launch(share=True, debug=False, show_error=True)

---
## C4 · Evaluación automática de la recuperación (hit@k)

`eval/preguntas.csv` tiene 10 preguntas con su fuente esperada y una palabra clave que debe aparecer en el chunk correcto. Como MarkItDown pega palabras, la comparación ignora espacios y mayúsculas.

In [26]:
import csv, io

REPO_RAW_BASE = "https://raw.githubusercontent.com/JeronimoCH/si3003-workshop-rag-JeronimoCH/main"


def cargar_preguntas(ruta: str = "eval/preguntas.csv"):
    for candidate in (Path(ruta), Path("..") / ruta):
        if candidate.exists():
            return list(csv.DictReader(candidate.open(encoding="utf-8")))
    response = requests.get(f"{REPO_RAW_BASE}/{ruta}", timeout=30)
    response.raise_for_status()
    return list(csv.DictReader(io.StringIO(response.text)))


def squash(text: str) -> str:
    return re.sub(r"\s+", "", text).lower()


def hit_at_k(retr, filas):
    hits = 0
    for r in filas:
        docs = retr.invoke(r["pregunta"])
        hits += any(
            squash(r["palabra_clave"]) in squash(d.page_content) and d.metadata["source"] == r["fuente_esperada"]
            for d in docs
        )
    return hits / len(filas)


filas = cargar_preguntas()
print(f"{len(filas)} preguntas\n")
print("| chunk_size | hit@3 | hit@5 |")
print("|---:|---:|---:|")
c4_rows = {}
for size in (400, 900, 1500):
    store, _ = stores[size]
    row = []
    for k in (3, 5):
        retr = store.as_retriever(search_kwargs={"k": k})   # sin umbral: medimos solo el ranking
        row.append(hit_at_k(retr, filas))
    c4_rows[size] = row
    print(f"| {size} | {row[0]:.0%} | {row[1]:.0%} |")

10 preguntas

| chunk_size | hit@3 | hit@5 |
|---:|---:|---:|


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


| 400 | 70% | 90% |
| 900 | 90% | 90% |
| 1500 | 90% | 100% |


In [27]:
# Detalle por pregunta con chunk_size=900 y k=5: ¿cuáles fallan?
retr = stores[900][0].as_retriever(search_kwargs={"k": 5})
for r in filas:
    docs = retr.invoke(r["pregunta"])
    ok = any(squash(r["palabra_clave"]) in squash(d.page_content) and d.metadata["source"] == r["fuente_esperada"] for d in docs)
    print(f"{'✔' if ok else '✘'} {r['id']:>2} {r['pregunta'][:70]:70s} → {[d.metadata['source'][:3] + '#' + str(d.metadata['chunk_index']) for d in docs]}")

✔  1 How many attention heads does the base Transformer use?                → ['att#14', 'att#31', 'att#40', 'att#9', 'att#39']
✔  2 What are the two RAG model variants?                                   → ['rag#10', 'rag#11', 'rag#24', 'rag#30', 'rag#40']
✔  3 What BLEU score did the big Transformer achieve on WMT 2014 English-to → ['att#28', 'att#1', 'att#26', 'att#31', 'att#24']
✔  4 How many warmup steps were used in the Transformer learning rate sched → ['att#29', 'att#25', 'att#31', 'att#30', 'att#28']
✘  5 What regularization techniques were used to train the Transformer?     → ['att#29', 'att#1', 'att#6', 'att#25', 'att#40']
✔  6 How long did it take to train the big Transformer model?               → ['att#28', 'att#29', 'att#25', 'att#24', 'att#1']
✔  7 How many documents does the Wikipedia index used by RAG contain?       → ['rag#86', 'rag#42', 'rag#18', 'rag#52', 'rag#7']
✔  8 Which approximation does RAG use in FAISS for fast retrieval?          → ['rag#86', 'rag#13', 'rag

---
## C5 · MMR frente a similitud pura

MMR toma `fetch_k=20` candidatos por similitud y elige 5 equilibrando relevancia y diversidad (`lambda_mult=0.5`).

In [28]:
sim_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
mmr_retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20, "lambda_mult": 0.5})


def redundancy(docs) -> float:
    """Similitud coseno media entre pares de chunks recuperados (más alto = más redundante)."""
    if len(docs) < 2:
        return 0.0
    vs = np.array(embeddings.embed_documents([d.page_content for d in docs]))
    vs = vs / np.linalg.norm(vs, axis=1, keepdims=True)
    sims = vs @ vs.T
    return float(sims[np.triu_indices(len(docs), 1)].mean())


C5_QUESTIONS = {"Q2 (amplia)": q2, "Q5 (dato puntual)": q5, "Bonus (dos papers)": q_bonus}
print("| Pregunta | Estrategia | Chunks | Papers distintos | Redundancia |")
print("|---|---|---|:---:|---:|")
for name, question in C5_QUESTIONS.items():
    for label, retr in (("similitud", sim_retriever), ("MMR", mmr_retriever)):
        docs = retr.invoke(question)
        ids = ", ".join(f"{d.metadata['source'][:3]}#{d.metadata['chunk_index']}" for d in docs)
        papers = len({d.metadata["source"] for d in docs})
        print(f"| {name} | {label} | {ids} | {papers} | {redundancy(docs):.3f} |")

| Pregunta | Estrategia | Chunks | Papers distintos | Redundancia |
|---|---|---|:---:|---:|


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


| Q2 (amplia) | similitud | rag#8, rag#1, rag#7, rag#32, rag#50 | 1 | 0.672 |


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


...

| Q5 (dato puntual) | MMR | att#14, att#31, att#0, att#35, att#40 | 1 | 0.434 |


| Bonus (dos papers) | similitud | rag#10, rag#12, rag#13, rag#86, rag#11 | 1 | 0.590 |


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


| Bonus (dos papers) | MMR | rag#10, att#0, rag#40, rag#86, rag#25 | 2 | 0.391 |


In [29]:
# Respuestas con cada estrategia para la pregunta bonus
mmr_chain = {"context": mmr_retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
sim_chain = {"context": sim_retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()
for label, chain in (("similitud", sim_chain), ("MMR", mmr_chain)):
    print(f"--- {label}\n{chain.invoke(q_bonus)}\n")
    time.sleep(1)

--- similitud
RAG uses a seq2seq generator based on the Transformer architecture, specifically a generator parametrized as p_θ(y_i | x, z, y_{1:i-1}) that generates the current token based on the previous tokens, the original input x, and the retrieved passage z. The paper states it can be initialized from a pre-trained bi-encoder from DPR and is implemented within HuggingFace Transformers.



  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 3


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 4


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 5


--- MMR
RAG uses a sequence-to-sequence model as its generator component, and this model is based on the Transformer architecture. The RAG paper states that the generator is parametrized as p_θ(y_i | x, z, y_{1:i-1}), and it compares favorably to T5, which is a Transformer-based model. The Attention paper proposes the Transformer architecture, and RAG leverages this architecture for its generator component.



---
## C6 · Persistencia del índice en disco

`QdrantClient(path=...)` guarda la colección en disco. La primera vez se indexa (y se pagan los embeddings); las siguientes se reabre sin llamar la API de embeddings.

**Prueba:** ejecuta esta celda, luego *Entorno de ejecución → Reiniciar sesión*, ejecuta solo las celdas de la Parte 0 y esta (no hace falta descargar ni convertir los PDFs si ya existe la colección) y compara los tiempos.

> Para que sobreviva a un *reinicio de máquina* de Colab (no solo del kernel), monta Google Drive y usa `PERSIST_PATH = "/content/drive/MyDrive/qdrant_data"`.

In [30]:
import time
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

PERSIST_PATH = "./qdrant_data"
PERSIST_COLLECTION = "academic_papers_nvidia"

emb_p = RetryEmbeddings(NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END"))
if "disk_client" in globals():
    disk_client.close()  # el modo local de Qdrant solo admite un cliente por carpeta

t0 = time.perf_counter()
disk_client = QdrantClient(path=PERSIST_PATH)
if disk_client.collection_exists(PERSIST_COLLECTION):
    mode = "reabrir"
else:
    mode = "indexar"
    dim = len(emb_p.embed_query("dimension probe"))
    disk_client.create_collection(PERSIST_COLLECTION, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
    QdrantVectorStore(client=disk_client, collection_name=PERSIST_COLLECTION, embedding=emb_p).add_documents(all_docs)
disk_store = QdrantVectorStore(client=disk_client, collection_name=PERSIST_COLLECTION, embedding=emb_p)
elapsed = time.perf_counter() - t0
count = disk_client.count(PERSIST_COLLECTION).count
print(f"Modo: {mode} · {count} puntos · {elapsed:.2f} s")

llm_p = ChatNVIDIA(
    model=GENERATION_MODEL,
    model_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=800,
).with_retry(stop_after_attempt=4)
prompt_p = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the supplied academic-paper excerpts. If the excerpts do not contain "
               "enough evidence, say so explicitly. Answer in the same language as the question and be "
               "concise. Mention the paper title supporting the answer."),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])
fmt = lambda docs: "\n\n---\n\n".join(f"[{d.metadata['title']} | {d.metadata['source']}]\n{d.page_content}" for d in docs)
disk_chain = ({"context": disk_store.as_retriever(search_kwargs={"k": 5}) | fmt, "question": RunnablePassthrough()}
              | prompt_p | llm_p | StrOutputParser())
print(disk_chain.invoke("What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"))

/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 2


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


Modo: indexar · 150 puntos · 87.20 s


28.4


### C6 · Reapertura tras reiniciar

La celda siguiente se ejecutó en un kernel nuevo, después de la celda de configuración de la Parte 0 y sin descargar ni indexar nada: reabre `./qdrant_data`.

In [2]:
# C6 · Ejecutada en un KERNEL NUEVO (solo la celda de configuración + esta), sin reindexar.
import time
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

PERSIST_PATH = "./qdrant_data"
PERSIST_COLLECTION = "academic_papers_nvidia"

emb_p = RetryEmbeddings(NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END"))
if "disk_client" in globals():
    disk_client.close()  # el modo local de Qdrant solo admite un cliente por carpeta

t0 = time.perf_counter()
disk_client = QdrantClient(path=PERSIST_PATH)
if disk_client.collection_exists(PERSIST_COLLECTION):
    mode = "reabrir"
else:
    mode = "indexar"
    dim = len(emb_p.embed_query("dimension probe"))
    disk_client.create_collection(PERSIST_COLLECTION, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
    QdrantVectorStore(client=disk_client, collection_name=PERSIST_COLLECTION, embedding=emb_p).add_documents(all_docs)
disk_store = QdrantVectorStore(client=disk_client, collection_name=PERSIST_COLLECTION, embedding=emb_p)
elapsed = time.perf_counter() - t0
count = disk_client.count(PERSIST_COLLECTION).count
print(f"Modo: {mode} · {count} puntos · {elapsed:.2f} s")

llm_p = ChatNVIDIA(
    model=GENERATION_MODEL,
    model_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=800,
).with_retry(stop_after_attempt=4)
prompt_p = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the supplied academic-paper excerpts. If the excerpts do not contain "
               "enough evidence, say so explicitly. Answer in the same language as the question and be "
               "concise. Mention the paper title supporting the answer."),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])
fmt = lambda docs: "\n\n---\n\n".join(f"[{d.metadata['title']} | {d.metadata['source']}]\n{d.page_content}" for d in docs)
disk_chain = ({"context": disk_store.as_retriever(search_kwargs={"k": 5}) | fmt, "question": RunnablePassthrough()}
              | prompt_p | llm_p | StrOutputParser())
print(disk_chain.invoke("What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"))

/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


  ↻ embeddings: HTTPSConnectionPool(host='inte · reintento 1


Modo: reabrir · 150 puntos · 14.03 s


28.4


---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.
